In [1]:
import pandas as pd

candidate = pd.read_csv(
    "../dataset/dummy/train/candidate_pairs.tsv",
    sep="\t"
)

print("Candidate rows:", len(candidate))

Candidate rows: 10000


In [2]:
source1 = pd.read_csv(
    "../dataset/dummy/train/source1.tsv",
    sep="\t"
)

print("Source 1 rows:", len(source1))

Source 1 rows: 1000


In [3]:
candidate = candidate[
    candidate["source1_entity_id"].isin(
        set(source1["entity_id"])
    )
].copy()

candidate_pairs = (
    candidate[
        ["source1_entity_id", "candidate_entity_ids"]
    ]
    .assign(
        candidate_entity_id=lambda x:
        x["candidate_entity_ids"].str.split(",")
    )
    .explode("candidate_entity_id")
)

candidate_pairs["candidate_entity_id"] = (
    candidate_pairs["candidate_entity_id"]
    .str.strip()
)

candidate_pairs = candidate_pairs.dropna(
    subset=["candidate_entity_id"]
).copy()

print("Candidate pairs:", len(candidate_pairs))

Candidate pairs: 47932


In [4]:
source2_path = "../dataset/train/train_source2.tsv"
source3_path = "../dataset/train/train_source3.tsv"

needed_ids = set(
    candidate_pairs["candidate_entity_id"]
)

def extract_records(path, needed_ids, chunksize=100_000):

    results = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):

        matched = chunk[
            chunk["entity_id"].isin(needed_ids)
        ]

        if len(matched) > 0:
            results.append(matched)

    if results:
        return pd.concat(
            results,
            ignore_index=True
        )

    return pd.DataFrame(
        columns=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    )


print("Reading Source 2...")
source2_needed = extract_records(
    source2_path,
    needed_ids
)

print("Source 2:", len(source2_needed))


print("Reading Source 3...")
source3_needed = extract_records(
    source3_path,
    needed_ids
)

print("Source 3:", len(source3_needed))


candidate_entities = pd.concat(
    [source2_needed, source3_needed],
    ignore_index=True
)

print("Total candidate entities:",
      len(candidate_entities))

Reading Source 2...
Source 2: 7310
Reading Source 3...
Source 3: 7306
Total candidate entities: 14616


In [5]:
# Rename Source 1 columns
source1 = source1.rename(columns={
    "entity_id": "source1_entity_id",
    "business_name": "business_name_s1",
    "business_address": "business_address_s1",
    "country": "country_s1"
})

# Rename candidate columns
candidate_entities = candidate_entities.rename(columns={
    "entity_id": "candidate_entity_id",
    "business_name": "business_name_candidate",
    "business_address": "business_address_candidate",
    "country": "country_candidate"
})

# Remove duplicate candidate IDs
candidate_entities = candidate_entities.drop_duplicates(
    subset=["candidate_entity_id"]
)

# Merge S1 information
pair_df = candidate_pairs.merge(
    source1,
    on="source1_entity_id",
    how="left"
)

# Merge S2/S3 information
pair_df = pair_df.merge(
    candidate_entities,
    on="candidate_entity_id",
    how="left"
)

print("Pair table shape:", pair_df.shape)
print(pair_df.head())

Pair table shape: (47932, 9)
  source1_entity_id                               candidate_entity_ids  \
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
1      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
2      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
3      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
4      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   

  candidate_entity_id     business_name_s1  \
0        S2-181962832  Orelee's Barbershop   
1        S2-185405451  Orelee's Barbershop   
2        S2-209305476  Orelee's Barbershop   
3        S2-372871261  Orelee's Barbershop   
4        S2-440253063  Orelee's Barbershop   

                      business_address_s1 country_s1  \
0  1795 Westchester Drive, High Point, NC         US   
1  1795 Westchester Drive, High Point, NC         US   
2  1795 Westchester Drive, High Point, NC         US   
3  1795 Westchester Drive

In [6]:
ground_truth = pd.read_csv(
    "../dataset/dummy/train/ground_truth.tsv",
    sep="\t"
)

true_matches = {}

for _, row in ground_truth.iterrows():

    s1_id = row["source1_entity_id"]

    if (
        pd.isna(row["matched_entity_ids"])
        or str(row["matched_entity_ids"]).strip() == ""
    ):
        true_matches[s1_id] = set()

    else:
        true_matches[s1_id] = set(
            x.strip()
            for x in str(
                row["matched_entity_ids"]
            ).split(",")
            if x.strip()
        )


pair_df["label"] = pair_df.apply(
    lambda row:
        int(
            row["candidate_entity_id"]
            in true_matches.get(
                row["source1_entity_id"],
                set()
            )
        ),
    axis=1
)

print("Pair table:", pair_df.shape)

print("\nLabels:")
print(pair_df["label"].value_counts())

Pair table: (47932, 10)

Labels:
label
0    47926
1        6
Name: count, dtype: int64


In [7]:
pair_df.to_csv(
    "../output/dev_pair_df.tsv",
    sep="\t",
    index=False
)

print("Saved development pair table.")

Saved development pair table.


In [8]:
# Imports
import sys
import os

import pandas as pd
import numpy as np

sys.path.append("..")

from src.features import create_pair_features
from src.model import (
    build_logistic_model,
    build_random_forest,
    train_model,
    predict_probabilities,
    save_model
)

from src.evaluation import (
    load_ground_truth,
    predictions_from_probabilities,
    calculate_entity_f05,
    find_best_threshold
)

In [9]:
# Create Features
X = create_pair_features(pair_df)

y = pair_df["label"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

X shape: (47932, 21)
y shape: (47932,)

Feature columns:
['name_ratio', 'name_partial_ratio', 'name_token_sort_ratio', 'name_token_set_ratio', 'name_jaccard', 'address_ratio', 'address_partial_ratio', 'address_token_sort_ratio', 'address_token_set_ratio', 'address_jaccard', 'name_length_diff', 'address_length_diff', 'name_missing_s1', 'name_missing_candidate', 'address_missing_s1', 'address_missing_candidate', 'country_equal', 'both_country_missing', 'name_address_mean', 'name_address_min', 'name_address_max']


In [10]:
# Split
from sklearn.model_selection import train_test_split

s1_ids = pair_df[
    "source1_entity_id"
].unique()

train_s1, valid_s1 = train_test_split(
    s1_ids,
    test_size=0.2,
    random_state=42
)

train_mask = pair_df[
    "source1_entity_id"
].isin(train_s1)

valid_mask = pair_df[
    "source1_entity_id"
].isin(valid_s1)

X_train = X.loc[train_mask]
X_valid = X.loc[valid_mask]

y_train = y.loc[train_mask]
y_valid = y.loc[valid_mask]

pair_train = pair_df.loc[
    train_mask
].copy()

pair_valid = pair_df.loc[
    valid_mask
].copy()

print("Training pairs:", len(X_train))
print("Validation pairs:", len(X_valid))

print(
    "Training S1s:",
    pair_train["source1_entity_id"].nunique()
)

print(
    "Validation S1s:",
    pair_valid["source1_entity_id"].nunique()
)

Training pairs: 37333
Validation pairs: 10599
Training S1s: 792
Validation S1s: 199


In [11]:
# Train Random Forest
rf_model = build_random_forest()

rf_model = train_model(
    rf_model,
    X_train,
    y_train
)

rf_probabilities = predict_probabilities(
    rf_model,
    X_valid
)

print("Random Forest trained.")

Random Forest trained.


In [12]:
# Load Ground Truth
ground_truth = load_ground_truth(
    "../dataset/dummy/train/ground_truth.tsv"
)

valid_s1_ids = pair_valid[
    "source1_entity_id"
].unique()

print(
    "Validation S1s:",
    len(valid_s1_ids)
)

Validation S1s: 199


In [13]:
# Find Random Forest Threshold
best_rf_threshold, best_rf_f05, rf_results = (
    find_best_threshold(
        pair_valid,
        rf_probabilities,
        ground_truth,
        valid_s1_ids
    )
)

print(
    "Best Random Forest threshold:",
    best_rf_threshold
)

print(
    "Best Random Forest F0.5:",
    best_rf_f05
)

Best Random Forest threshold: 0.05
Best Random Forest F0.5: 0.053042992741485204


In [14]:
# Train Final Model
final_model = build_random_forest()

final_model = train_model(
    final_model,
    X,
    y
)

In [15]:
# Next Start

In [16]:
# Import Everything
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("..")

from src.features import create_pair_features
from src.model import load_model, predict_probabilities
from src.submission import (
    explode_candidate_file,
    generate_submission,
    validate_prediction_subset
)

print("Imports successful")

Imports successful


In [17]:
# Load the saved final model
model_path = "../output/final_model.joblib"

final_model = load_model(model_path)

print("Final model loaded:")
print(final_model)

Final model loaded:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=2, n_estimators=300,
                                        n_jobs=-1, random_state=42))])


In [18]:
# Load the selected threshold
best_threshold = 0.05   # replace XX with your selected threshold
print("Using threshold:", best_threshold)

Using threshold: 0.05


In [19]:
# Load test source 1
test_source1 = pd.read_csv(
    "../dataset/test/test_source1.tsv",
    sep="\t"
)

print("Test Source 1 shape:", test_source1.shape)
print(test_source1.head())

Test Source 1 shape: (1732544, 4)
      entity_id         business_name  \
0  S1-714132312       Zephay Labs Inc   
1  S1-106407869  Vision Partners Corp   
2  S1-156285671         << Team Ecole   
3  S1-689823050   Red Perfect Trading   
4  S1-921369899         ZNB Club SARL   

                                    business_address country  
0                        2621 Cotten Road, Tyler, TX      US  
1             IA, Iowa City, 1064 Newton Rd, Unit 11      US  
2  175 Boulevard du Président Franklin Roosevelt,...  France  
3  Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar...   India  
4  Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Ru...  France  


In [20]:
# Load the test candidate file
candidate_file = "../output/candidate_pairs.tsv"

candidate = pd.read_csv(
    candidate_file,
    sep="\t"
)

print("Candidate file shape:", candidate.shape)
print(candidate.head())

Candidate file shape: (10000, 2)
  source1_entity_id                               candidate_entity_ids
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...
1      S1-773889195  S2-145204119,S2-150634918,S2-1869428,S2-218203...
2      S1-377745466  S2-131264573,S2-170803924,S2-52172500,S2-54191...
3      S1-133037285  S2-154192027,S2-241726445,S2-247565679,S2-2620...
4      S1-755362802  S2-10068244,S2-102907056,S2-106571440,S2-18105...


In [21]:
# Keep only TEST Source 1 IDs
test_s1_ids = set(test_source1["entity_id"])

candidate = candidate[
    candidate["source1_entity_id"].isin(test_s1_ids)
].copy()

print("Candidate rows for test:", len(candidate))

Candidate rows for test: 0


In [22]:
print("Test S1 sample:")
print(test_source1["entity_id"].head().tolist())

print("\nCandidate S1 sample:")
print(candidate["source1_entity_id"].head().tolist())

Test S1 sample:
['S1-714132312', 'S1-106407869', 'S1-156285671', 'S1-689823050', 'S1-921369899']

Candidate S1 sample:
[]


In [23]:
print("Number of test S1 IDs:", len(test_s1_ids))
print("Number of candidate S1 IDs:", candidate["source1_entity_id"].nunique())

Number of test S1 IDs: 1732544
Number of candidate S1 IDs: 0


In [24]:
print(
    "Overlap:",
    len(
        set(test_source1["entity_id"])
        &
        set(candidate["source1_entity_id"])
    )
)

Overlap: 0


In [25]:
candidate = pd.read_csv(
    "../output/candidate_pairs.tsv",
    sep="\t"
)

print(candidate.shape)
print(candidate.columns)

print("\nUnique S1 IDs:")
print(candidate["source1_entity_id"].nunique())

print("\nFirst 10 S1 IDs:")
print(candidate["source1_entity_id"].head(10).tolist())

(10000, 2)
Index(['source1_entity_id', 'candidate_entity_ids'], dtype='str')

Unique S1 IDs:
10000

First 10 S1 IDs:
['S1-925783039', 'S1-773889195', 'S1-377745466', 'S1-133037285', 'S1-755362802', 'S1-851869949', 'S1-785847572', 'S1-27541239', 'S1-629417405', 'S1-22305073']


In [26]:
import pandas as pd

# Load dummy training Source 1
source1 = pd.read_csv(
    "../dataset/dummy/train/source1.tsv",
    sep="\t"
)

# Load your candidate file
candidate = pd.read_csv(
    "../output/candidate_pairs.tsv",
    sep="\t"
)

print("Dummy Source 1:", source1.shape)
print("Candidate file:", candidate.shape)

# Keep only candidate rows belonging to our dummy Source 1 IDs
candidate = candidate[
    candidate["source1_entity_id"].isin(source1["entity_id"])
].copy()

print("Candidate rows for dummy data:", candidate.shape)
print("Unique S1 IDs:", candidate["source1_entity_id"].nunique())

Dummy Source 1: (1000, 4)
Candidate file: (10000, 2)
Candidate rows for dummy data: (1000, 2)
Unique S1 IDs: 1000


In [27]:
candidate_pairs = (
    candidate[["source1_entity_id", "candidate_entity_ids"]]
    .assign(
        candidate_entity_id=lambda x:
            x["candidate_entity_ids"].str.split(",")
    )
    .explode("candidate_entity_id")
)

candidate_pairs["candidate_entity_id"] = (
    candidate_pairs["candidate_entity_id"].str.strip()
)

candidate_pairs = candidate_pairs.dropna(
    subset=["candidate_entity_id"]
).copy()

print("Candidate pairs:", candidate_pairs.shape)
print(candidate_pairs.head())

Candidate pairs: (47932, 3)
  source1_entity_id                               candidate_entity_ids  \
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   
0      S1-925783039  S2-181962832,S2-185405451,S2-209305476,S2-3728...   

  candidate_entity_id  
0        S2-181962832  
0        S2-185405451  
0        S2-209305476  
0        S2-372871261  
0        S2-440253063  


In [28]:
candidate_pairs = candidate_pairs[
    ["source1_entity_id", "candidate_entity_id"]
].copy()

print("Candidate pairs:", candidate_pairs.shape)
print(candidate_pairs.head())

Candidate pairs: (47932, 2)
  source1_entity_id candidate_entity_id
0      S1-925783039        S2-181962832
0      S1-925783039        S2-185405451
0      S1-925783039        S2-209305476
0      S1-925783039        S2-372871261
0      S1-925783039        S2-440253063


In [29]:
# Start
# Part A: Build the Pair Dataframe

In [30]:
# Get Candidate IDs
needed_ids = set(
    candidate_pairs["candidate_entity_id"].dropna()
)

print("Unique candidate IDs needed:", len(needed_ids))

Unique candidate IDs needed: 14616


In [31]:
# Function to extract only required records
def extract_records(path, needed_ids, chunksize=100_000):
    results = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):
        matched = chunk[
            chunk["entity_id"].isin(needed_ids)
        ]

        if len(matched) > 0:
            results.append(matched)

    if results:
        return pd.concat(
            results,
            ignore_index=True
        )

    return pd.DataFrame(
        columns=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    )

In [32]:
# Extract Source 2 candidates
source2_needed = extract_records(
    "../dataset/train/train_source2.tsv",
    needed_ids
)

print(
    "Source 2 candidate records:",
    source2_needed.shape
)

Source 2 candidate records: (7310, 4)


In [33]:
# Extract Source 3 candidate
source3_needed = extract_records(
    "../dataset/train/train_source3.tsv",
    needed_ids
)

print(
    "Source 3 candidate records:",
    source3_needed.shape
)

Source 3 candidate records: (7306, 4)


In [34]:
# Combine S2 + S3
candidate_entities = pd.concat(
    [source2_needed, source3_needed],
    ignore_index=True
)

candidate_entities = candidate_entities.drop_duplicates(
    subset=["entity_id"]
)

print(
    "Total candidate entity records:",
    candidate_entities.shape
)

Total candidate entity records: (14616, 4)


In [35]:
# Prepare Source 1
s1_for_merge = source1.rename(
    columns={
        "entity_id": "source1_entity_id",
        "business_name": "business_name_s1",
        "business_address": "business_address_s1",
        "country": "country_s1"
    }
)

print(s1_for_merge.head())

  source1_entity_id         business_name_s1  \
0      S1-925783039      Orelee's Barbershop   
1      S1-773889195              Prime Money   
2      S1-377745466            B+ Retail Inc   
3      S1-133037285            Christ Chapel   
4      S1-755362802  Prabhav Business Center   

                                 business_address_s1 country_s1  
0             1795 Westchester Drive, High Point, NC         US  
1                    17560 Ellis Road, Tahlequah, OK         US  
2                1712 Montebello Avenue, Phoenix, AZ         US  
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD         US  
4  797, Lake Town Block A, Kolkata, Howrah, West ...      India  


In [36]:
# Prepare candidate entities
candidate_for_merge = candidate_entities.rename(
    columns={
        "entity_id": "candidate_entity_id",
        "business_name": "business_name_candidate",
        "business_address": "business_address_candidate",
        "country": "country_candidate"
    }
)

print(candidate_for_merge.head())

  candidate_entity_id          business_name_candidate  \
0        S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1        S2-764573417     -- Holloway Peak Inc Seafood   
2        S2-639257739         आदित्य प्रॉपर्टीज एलएलपी   
3        S2-163963287                       Summit Inc   
4         S2-49942811     Delta Tetlecommunication Inc   

                         business_address_candidate country_candidate  
0      KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi             India  
1                         105 ELM ST, MORGANTON, NC                US  
2  G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh             India  
3             GREENSBORO, NC, 19 1/2 STARDUST TRAIL                US  
4                   914 PIERPONT AVE, CLEVELAND, OH                US  


In [37]:
# Build pair_df
pair_df = candidate_pairs.merge(
    s1_for_merge,
    on="source1_entity_id",
    how="left"
)

pair_df = pair_df.merge(
    candidate_for_merge,
    on="candidate_entity_id",
    how="left"
)

print("pair_df shape:", pair_df.shape)
print(pair_df.head())

pair_df shape: (47932, 8)
  source1_entity_id candidate_entity_id     business_name_s1  \
0      S1-925783039        S2-181962832  Orelee's Barbershop   
1      S1-925783039        S2-185405451  Orelee's Barbershop   
2      S1-925783039        S2-209305476  Orelee's Barbershop   
3      S1-925783039        S2-372871261  Orelee's Barbershop   
4      S1-925783039        S2-440253063  Orelee's Barbershop   

                      business_address_s1 country_s1  \
0  1795 Westchester Drive, High Point, NC         US   
1  1795 Westchester Drive, High Point, NC         US   
2  1795 Westchester Drive, High Point, NC         US   
3  1795 Westchester Drive, High Point, NC         US   
4  1795 Westchester Drive, High Point, NC         US   

        business_name_candidate          business_address_candidate  \
0  Vision Clinic of Greene Corp    038 HIGH SCHOOL BLVD, GREENE, IA   
1           Chiropractic Center   HIGH POINT, 106 ROCKSPRING RD, NC   
2       Novus Great (ID: 72515)        

In [38]:
# Check missing candidate records
missing = pair_df[
    pair_df["business_name_candidate"].isna()
]

print("Missing candidate records:", len(missing))

Missing candidate records: 0


In [39]:
# PART B — Add ground-truth labels

In [40]:
# Load ground truth
ground_truth = pd.read_csv(
    "../dataset/dummy/train/ground_truth.tsv",
    sep="\t"
)

print(ground_truth.shape)
print(ground_truth.head())

(1000, 2)
  source1_entity_id                                 matched_entity_ids
0      S1-844896591                           S2-362218588,S3-11966366
1      S1-548116192                                                NaN
2      S1-378978603             S2-319300693,S3-948532405,S3-921016187
3      S1-447452795                          S2-995233298,S3-366674728
4      S1-401761505  S2-422370961,S2-415623223,S3-17464132,S3-16723...


In [41]:
# Create true-match dictionary
true_matches = {}

for _, row in ground_truth.iterrows():
    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]

    if pd.isna(matched) or str(matched).strip() == "":
        true_matches[s1_id] = set()
    else:
        true_matches[s1_id] = set(
            str(matched).split(",")
        )

print("Ground-truth S1s:", len(true_matches))

Ground-truth S1s: 1000


In [42]:
# create label
pair_df["label"] = pair_df.apply(
    lambda row: int(
        row["candidate_entity_id"]
        in true_matches.get(
            row["source1_entity_id"],
            set()
        )
    ),
    axis=1
)

print(
    pair_df["label"].value_counts()
)

label
0    47926
1        6
Name: count, dtype: int64


In [43]:
# PART C — Save the development pair table
pair_df.to_csv(
    "../output/dev_pair_df.tsv",
    sep="\t",
    index=False
)

print("Development pair table saved.")

Development pair table saved.


In [44]:
# Feature Engineering
from src.features import create_pair_features

X = create_pair_features(pair_df)

y = pair_df["label"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (47932, 21)
y shape: (47932,)


In [45]:
# PART E — Train/validation split by S1
from sklearn.model_selection import train_test_split

all_s1_ids = pair_df["source1_entity_id"].unique()

train_s1, val_s1 = train_test_split(
    all_s1_ids,
    test_size=0.2,
    random_state=42
)

print("Training S1s:", len(train_s1))
print("Validation S1s:", len(val_s1))

Training S1s: 792
Validation S1s: 199


In [46]:
train_mask = pair_df["source1_entity_id"].isin(train_s1)
val_mask = pair_df["source1_entity_id"].isin(val_s1)

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]

y_train = y.loc[train_mask]
y_val = y.loc[val_mask]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (37333, 21)
X_val: (10599, 21)
y_train: (37333,)
y_val: (10599,)


In [47]:
# PART F — Train your model
from src.model import (
    build_random_forest,
    train_model,
    predict_probabilities,
    save_model
)

model = build_random_forest()

print(model)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=2, n_estimators=300,
                                        n_jobs=-1, random_state=42))])


In [48]:
model = train_model(
    model,
    X_train,
    y_train
)

print("Model training complete.")

Model training complete.


In [49]:
# Validation probabilities
val_probabilities = predict_probabilities(
    model,
    X_val
)

print(
    "Validation predictions:",
    len(val_probabilities)
)

print(
    "Probability range:",
    val_probabilities.min(),
    "to",
    val_probabilities.max()
)

Validation predictions: 10599
Probability range: 0.0 to 0.656635000197999


In [50]:
# PART G — Find the F0.5 threshold
from src.evaluation import (
    calculate_entity_f05,
    find_best_threshold
)

In [51]:
val_pair_df = pair_df.loc[
    val_mask
].copy()

val_pair_df = val_pair_df.reset_index(
    drop=True
)

y_val = y_val.reset_index(
    drop=True
)

In [52]:
# Find threashold
best_threshold, best_score, threshold_results = (
    find_best_threshold(
        val_pair_df,
        val_probabilities,
        ground_truth,
        val_s1
    )
)

print("Best threshold:", best_threshold)
print("Best validation F0.5:", best_score)

Best threshold: 0.6600000000000001
Best validation F0.5: 1.0


In [53]:
threshold_results.sort_values(
    "f05",
    ascending=False
).head(10)

,threshold,f05
64,0.69,1.0
65,0.70,1.0
66,0.71,1.0
67,0.72,1.0
68,0.73,1.0
69,0.74,1.0
70,0.75,1.0
71,0.76,1.0
72,0.77,1.0
73,0.78,1.0


In [54]:
# PART H — Train final model on all development data
final_model = build_random_forest()

final_model = train_model(
    final_model,
    X,
    y
)

print("Final model trained on all development data.")

Final model trained on all development data.


In [55]:
# Save final model
final_model_path = "../output/final_model.joblib"

save_model(
    final_model,
    final_model_path
)

print(
    "Final model saved:",
    final_model_path
)

Final model saved: ../output/final_model.joblib


In [56]:
# PART I — Create a dummy "test" prediction
# Get validation predictions
val_predictions = val_pair_df.loc[
    val_probabilities >= best_threshold,
    [
        "source1_entity_id",
        "candidate_entity_id"
    ]
].copy()

print(
    "Predicted matching pairs:",
    len(val_predictions)
)

print(val_predictions.head(20))

Predicted matching pairs: 0
Empty DataFrame
Columns: [source1_entity_id, candidate_entity_id]
Index: []


In [57]:
# Generate matching results
prediction_dict = (
    val_predictions
    .groupby("source1_entity_id")[
        "candidate_entity_id"
    ]
    .apply(list)
    .to_dict()
)

validation_submission = pd.DataFrame({
    "source1_entity_id": val_s1,
    "matched_entity_ids": [
        ",".join(
            prediction_dict.get(
                s1,
                []
            )
        )
        for s1 in val_s1
    ]
})

print(validation_submission.head(20))

   source1_entity_id matched_entity_ids
0       S1-896836973                   
1       S1-580254632                   
2       S1-607646145                   
3       S1-284710338                   
4       S1-970972284                   
5       S1-418896338                   
6        S1-31205819                   
7       S1-448166395                   
8       S1-666876590                   
9       S1-806365059                   
10      S1-977704542                   
11      S1-314045617                   
12      S1-684656443                   
13      S1-761874846                   
14      S1-198728897                   
15      S1-666435696                   
16      S1-710147242                   
17      S1-460033462                   
18      S1-223600564                   
19      S1-143265439                   


In [58]:
validation_submission_path = (
    "../output/matching_results.tsv"
)

validation_submission.to_csv(
    validation_submission_path,
    sep="\t",
    index=False
)

print(
    "matching_results.tsv saved!"
)

matching_results.tsv saved!


In [59]:
from src.submission import validate_prediction_subset

validate_prediction_subset(
    validation_submission,
    candidate_pairs[
        candidate_pairs["source1_entity_id"].isin(val_s1)
    ]
)

print("Prediction subset validation passed.")

Prediction subset validation passed.


In [60]:
# Final Sanity Check
submission = pd.read_csv(
    "../output/matching_results.tsv",
    sep="\t",
    dtype=str
)

print("========== FINAL CHECK ==========")

print("Submission shape:", submission.shape)

print(
    "Unique S1 IDs:",
    submission["source1_entity_id"].nunique()
)

print(
    "Rows with predictions:",
    submission["matched_entity_ids"]
    .fillna("")
    .str.strip()
    .ne("")
    .sum()
)

print(
    "Rows with no predictions:",
    submission["matched_entity_ids"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("\nFirst 10 rows:")
print(submission.head(10))

print("\nSelected threshold:", best_threshold)
print("Validation F0.5:", best_score)

========== FINAL CHECK ==========
Submission shape: (199, 2)
Unique S1 IDs: 199
Rows with predictions: 0
Rows with no predictions: 199

First 10 rows:
  source1_entity_id matched_entity_ids
0      S1-896836973                NaN
1      S1-580254632                NaN
2      S1-607646145                NaN
3      S1-284710338                NaN
4      S1-970972284                NaN
5      S1-418896338                NaN
6       S1-31205819                NaN
7      S1-448166395                NaN
8      S1-666876590                NaN
9      S1-806365059                NaN

Selected threshold: 0.6600000000000001
Validation F0.5: 1.0
